In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    when,
    lit,
    round,
    count,
    sum,
    avg,
    max,
    min
)

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
spark = (
    SparkSession.builder
    .appName("Spark Assignment")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark Session Created")
print("Spark Version :", spark.version)
print("Application Name :", spark.sparkContext.appName)
print("Master :", spark.sparkContext.master)

Spark Session Created
Spark Version : 3.5.1
Application Name : Spark Assignment
Master : local[*]


In [4]:
file_path = "../dataset/superstore.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(file_path)
)

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [5]:
df.show(10, truncate=False)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+----------------------------------------------------------------+--------+--------+--------+--------+
|Row ID|Order ID      |Order Date|Ship Date |Ship Mode     |Customer ID|Customer Name  |Segment  |Country      |City           |State     |Postal Code|Region|Product ID     |Category       |Sub-Category|Product Name                                                    |Sales   |Quantity|Discount|Profit  |
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+----------------------------------------------------------------+--------+--------+--------+--------+
|1     |CA-2016-152156|11/8/2016 |11/11/2016|Second Class  |CG-12520   |Claire Gute  

In [6]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [7]:
print("Total Rows :", df.count())
print("Total Columns :", len(df.columns))

Total Rows : 9994
Total Columns : 21


In [8]:
print("Columns in the dataset:\n")

for column in df.columns:
    print(column)

Columns in the dataset:

Row ID
Order ID
Order Date
Ship Date
Ship Mode
Customer ID
Customer Name
Segment
Country
City
State
Postal Code
Region
Product ID
Category
Sub-Category
Product Name
Sales
Quantity
Discount
Profit


In [9]:
selected_df = df.select(
    "Order ID",
    "Customer Name",
    "Category",
    "Sub-Category",
    "Sales",
    "Quantity",
    "Profit",
    "City",
    "State"
)

selected_df.show(10, truncate=False)

+--------------+---------------+---------------+------------+--------+--------+--------+---------------+----------+
|Order ID      |Customer Name  |Category       |Sub-Category|Sales   |Quantity|Profit  |City           |State     |
+--------------+---------------+---------------+------------+--------+--------+--------+---------------+----------+
|CA-2016-152156|Claire Gute    |Furniture      |Bookcases   |261.96  |2       |41.9136 |Henderson      |Kentucky  |
|CA-2016-152156|Claire Gute    |Furniture      |Chairs      |731.94  |3       |219.582 |Henderson      |Kentucky  |
|CA-2016-138688|Darrin Van Huff|Office Supplies|Labels      |14.62   |2       |6.8714  |Los Angeles    |California|
|US-2015-108966|Sean O'Donnell |Furniture      |Tables      |957.5775|5       |-383.031|Fort Lauderdale|Florida   |
|US-2015-108966|Sean O'Donnell |Office Supplies|Storage     |22.368  |2       |2.5164  |Fort Lauderdale|Florida   |
|CA-2014-115812|Brosina Hoffman|Furniture      |Furnishings |48.86   |7 

In [10]:
from pyspark.sql.functions import col

df = (
    df.withColumn("Sales", col("Sales").cast("double"))
      .withColumn("Quantity", col("Quantity").cast("int"))
      .withColumn("Discount", col("Discount").cast("double"))
)

print("Data types updated successfully.")

df.printSchema()

Data types updated successfully.
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [11]:
selected_df = df.select(
    "Order ID",
    "Customer Name",
    "Category",
    "Sub-Category",
    "Sales",
    "Quantity",
    "Profit",
    "City",
    "State"
)

selected_df.printSchema()
selected_df.show(5)

root
 |-- Order ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Profit: double (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)

+--------------+---------------+---------------+------------+--------+--------+--------+---------------+----------+
|      Order ID|  Customer Name|       Category|Sub-Category|   Sales|Quantity|  Profit|           City|     State|
+--------------+---------------+---------------+------------+--------+--------+--------+---------------+----------+
|CA-2016-152156|    Claire Gute|      Furniture|   Bookcases|  261.96|       2| 41.9136|      Henderson|  Kentucky|
|CA-2016-152156|    Claire Gute|      Furniture|      Chairs|  731.94|       3| 219.582|      Henderson|  Kentucky|
|CA-2016-138688|Darrin Van Huff|Office Supplies|      La

In [12]:
filtered_df = selected_df.filter(col("Sales") > 500)

filtered_df.show(10, truncate=False)

+--------------+---------------+---------------+------------+--------+--------+----------+---------------+------------+
|Order ID      |Customer Name  |Category       |Sub-Category|Sales   |Quantity|Profit    |City           |State       |
+--------------+---------------+---------------+------------+--------+--------+----------+---------------+------------+
|CA-2016-152156|Claire Gute    |Furniture      |Chairs      |731.94  |3       |219.582   |Henderson      |Kentucky    |
|US-2015-108966|Sean O'Donnell |Furniture      |Tables      |957.5775|5       |-383.031  |Fort Lauderdale|Florida     |
|CA-2014-115812|Brosina Hoffman|Technology     |Phones      |907.152 |6       |90.7152   |Los Angeles    |California  |
|CA-2014-115812|Brosina Hoffman|Furniture      |Tables      |1706.184|9       |85.3092   |Los Angeles    |California  |
|CA-2014-115812|Brosina Hoffman|Technology     |Phones      |911.424 |4       |68.3568   |Los Angeles    |California  |
|CA-2014-105893|Pete Kriz      |Office S

In [13]:
renamed_df = (
    filtered_df
    .withColumnRenamed("Customer Name", "Customer_Name")
    .withColumnRenamed("Sub-Category", "Sub_Category")
)

renamed_df.show(10, truncate=False)

+--------------+---------------+---------------+------------+--------+--------+----------+---------------+------------+
|Order ID      |Customer_Name  |Category       |Sub_Category|Sales   |Quantity|Profit    |City           |State       |
+--------------+---------------+---------------+------------+--------+--------+----------+---------------+------------+
|CA-2016-152156|Claire Gute    |Furniture      |Chairs      |731.94  |3       |219.582   |Henderson      |Kentucky    |
|US-2015-108966|Sean O'Donnell |Furniture      |Tables      |957.5775|5       |-383.031  |Fort Lauderdale|Florida     |
|CA-2014-115812|Brosina Hoffman|Technology     |Phones      |907.152 |6       |90.7152   |Los Angeles    |California  |
|CA-2014-115812|Brosina Hoffman|Furniture      |Tables      |1706.184|9       |85.3092   |Los Angeles    |California  |
|CA-2014-115812|Brosina Hoffman|Technology     |Phones      |911.424 |4       |68.3568   |Los Angeles    |California  |
|CA-2014-105893|Pete Kriz      |Office S

In [14]:
from pyspark.sql.functions import round

updated_df = renamed_df.withColumn(
    "Total_Amount",
    round(col("Sales") * col("Quantity"), 2)
)

updated_df.show(10, truncate=False)

+--------------+---------------+---------------+------------+--------+--------+----------+---------------+------------+------------+
|Order ID      |Customer_Name  |Category       |Sub_Category|Sales   |Quantity|Profit    |City           |State       |Total_Amount|
+--------------+---------------+---------------+------------+--------+--------+----------+---------------+------------+------------+
|CA-2016-152156|Claire Gute    |Furniture      |Chairs      |731.94  |3       |219.582   |Henderson      |Kentucky    |2195.82     |
|US-2015-108966|Sean O'Donnell |Furniture      |Tables      |957.5775|5       |-383.031  |Fort Lauderdale|Florida     |4787.89     |
|CA-2014-115812|Brosina Hoffman|Technology     |Phones      |907.152 |6       |90.7152   |Los Angeles    |California  |5442.91     |
|CA-2014-115812|Brosina Hoffman|Furniture      |Tables      |1706.184|9       |85.3092   |Los Angeles    |California  |15355.66    |
|CA-2014-115812|Brosina Hoffman|Technology     |Phones      |911.424 

In [15]:
from pyspark.sql.functions import when

status_df = updated_df.withColumn(
    "Order_Category",
    when(col("Total_Amount") >= 1000, "High Value")
    .otherwise("Regular")
)

status_df.show(10, truncate=False)

+--------------+---------------+---------------+------------+--------+--------+----------+---------------+------------+------------+--------------+
|Order ID      |Customer_Name  |Category       |Sub_Category|Sales   |Quantity|Profit    |City           |State       |Total_Amount|Order_Category|
+--------------+---------------+---------------+------------+--------+--------+----------+---------------+------------+------------+--------------+
|CA-2016-152156|Claire Gute    |Furniture      |Chairs      |731.94  |3       |219.582   |Henderson      |Kentucky    |2195.82     |High Value    |
|US-2015-108966|Sean O'Donnell |Furniture      |Tables      |957.5775|5       |-383.031  |Fort Lauderdale|Florida     |4787.89     |High Value    |
|CA-2014-115812|Brosina Hoffman|Technology     |Phones      |907.152 |6       |90.7152   |Los Angeles    |California  |5442.91     |High Value    |
|CA-2014-115812|Brosina Hoffman|Furniture      |Tables      |1706.184|9       |85.3092   |Los Angeles    |Califo

In [16]:
from pyspark.sql.functions import col, sum

status_df.select([
    sum(col(column).isNull().cast("int")).alias(column)
    for column in status_df.columns
]).show()

+--------+-------------+--------+------------+-----+--------+------+----+-----+------------+--------------+
|Order ID|Customer_Name|Category|Sub_Category|Sales|Quantity|Profit|City|State|Total_Amount|Order_Category|
+--------+-------------+--------+------------+-----+--------+------+----+-----+------------+--------------+
|       0|            0|       0|           0|    0|       0|     0|   0|    0|           0|             0|
+--------+-------------+--------+------------+-----+--------+------+----+-----+------------+--------------+



In [17]:
clean_df = status_df.fillna({
    "Customer_Name": "Unknown",
    "Category": "Not Available",
    "Sub_Category": "Not Available",
    "City": "Unknown",
    "State": "Unknown"
})

clean_df.show(10, truncate=False)

+--------------+---------------+---------------+------------+--------+--------+----------+---------------+------------+------------+--------------+
|Order ID      |Customer_Name  |Category       |Sub_Category|Sales   |Quantity|Profit    |City           |State       |Total_Amount|Order_Category|
+--------------+---------------+---------------+------------+--------+--------+----------+---------------+------------+------------+--------------+
|CA-2016-152156|Claire Gute    |Furniture      |Chairs      |731.94  |3       |219.582   |Henderson      |Kentucky    |2195.82     |High Value    |
|US-2015-108966|Sean O'Donnell |Furniture      |Tables      |957.5775|5       |-383.031  |Fort Lauderdale|Florida     |4787.89     |High Value    |
|CA-2014-115812|Brosina Hoffman|Technology     |Phones      |907.152 |6       |90.7152   |Los Angeles    |California  |5442.91     |High Value    |
|CA-2014-115812|Brosina Hoffman|Furniture      |Tables      |1706.184|9       |85.3092   |Los Angeles    |Califo

In [18]:
category_sales = (
    clean_df.groupBy("Category")
    .sum("Sales")
    .orderBy("sum(Sales)", ascending=False)
)

category_sales.show()

+---------------+------------------+
|       Category|        sum(Sales)|
+---------------+------------------+
|     Technology| 605739.5919999996|
|      Furniture|490694.14890000015|
|Office Supplies| 371333.7640000001|
+---------------+------------------+



In [19]:
import sys

print("Python Executable:")
print(sys.executable)

print("\nPython Version:")
print(sys.version)

Python Executable:
e:\celabal\spark-data-engineering-pipeline\venv\Scripts\python.exe

Python Version:
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [20]:
import pyspark

print(pyspark.__version__)

3.5.1


In [22]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Spark Assignment")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark Version :", spark.version)
print("Application Name :", spark.sparkContext.appName)

Spark Version : 3.5.1
Application Name : Spark Assignment


In [23]:
lazy_df = (
    clean_df
    .filter(col("Sales") > 500)
    .select(
        "Order ID",
        "Customer_Name",
        "Category",
        "Sales",
        "Profit",
        "Order_Category"
    )
)

print("Execution Plan")
lazy_df.explain(True)

Execution Plan
== Parsed Logical Plan ==
'Project ['Order ID, 'Customer_Name, 'Category, 'Sales, 'Profit, 'Order_Category]
+- Filter (Sales#247 > cast(500 as double))
   +- Project [Order ID#18, coalesce(Customer_Name#414, cast(Unknown as string)) AS Customer_Name#747, coalesce(Category#31, cast(Not Available as string)) AS Category#748, coalesce(Sub_Category#424, cast(Not Available as string)) AS Sub_Category#749, Sales#247, Quantity#269, Profit#37, coalesce(City#26, cast(Unknown as string)) AS City#750, coalesce(State#27, cast(Unknown as string)) AS State#751, Total_Amount#480, Order_Category#541]
      +- Project [Order ID#18, Customer_Name#414, Category#31, Sub_Category#424, Sales#247, Quantity#269, Profit#37, City#26, State#27, Total_Amount#480, CASE WHEN (Total_Amount#480 >= cast(1000 as double)) THEN High Value ELSE Regular END AS Order_Category#541]
         +- Project [Order ID#18, Customer_Name#414, Category#31, Sub_Category#424, Sales#247, Quantity#269, Profit#37, City#26, S

In [24]:
lazy_df.show(10, truncate=False)

+--------------+---------------+---------------+--------+----------+--------------+
|Order ID      |Customer_Name  |Category       |Sales   |Profit    |Order_Category|
+--------------+---------------+---------------+--------+----------+--------------+
|CA-2016-152156|Claire Gute    |Furniture      |731.94  |219.582   |High Value    |
|US-2015-108966|Sean O'Donnell |Furniture      |957.5775|-383.031  |High Value    |
|CA-2014-115812|Brosina Hoffman|Technology     |907.152 |90.7152   |High Value    |
|CA-2014-115812|Brosina Hoffman|Furniture      |1706.184|85.3092   |High Value    |
|CA-2014-115812|Brosina Hoffman|Technology     |911.424 |68.3568   |High Value    |
|CA-2014-105893|Pete Kriz      |Office Supplies|665.88  |13.3176   |High Value    |
|CA-2015-106320|Emily Burns    |Furniture      |1044.63 |240.2649  |High Value    |
|US-2015-150630|Tracy Blumstein|Furniture      |3083.43 |-1665.0522|High Value    |
|CA-2016-117590|Gene Hale      |Technology     |1097.544|123.4737  |High Val

In [28]:
print("Java Version:", spark.sparkContext._jvm.java.lang.System.getProperty("java.version"))
print("Java Home:", spark.sparkContext._jvm.java.lang.System.getProperty("java.home"))

Java Version: 22.0.1
Java Home: C:\Program Files\Java\jdk-22


In [30]:
import os

os.makedirs(r"E:\celabal\spark-data-engineering-pipeline\output\csv_output", exist_ok=True)

clean_df.toPandas().to_csv(
    r"E:\celabal\spark-data-engineering-pipeline\output\csv_output\final_output.csv",
    index=False
)

print("CSV saved successfully.")

CSV saved successfully.


In [31]:
os.makedirs(r"E:\celabal\spark-data-engineering-pipeline\output\parquet_output", exist_ok=True)

clean_df.toPandas().to_parquet(
    r"E:\celabal\spark-data-engineering-pipeline\output\parquet_output\final_output.parquet",
    index=False
)

print("Parquet saved successfully.")

Parquet saved successfully.


In [32]:
import os

output_path = r"E:\celabal\spark-data-engineering-pipeline\output"

os.makedirs(output_path, exist_ok=True)

clean_df.toPandas().to_csv(
    os.path.join(output_path, "superstore_output.csv"),
    index=False
)

print("CSV saved successfully.")

CSV saved successfully.


In [33]:
clean_df.toPandas().to_parquet(
    os.path.join(output_path, "superstore_output.parquet"),
    index=False
)

print("Parquet saved successfully.")

Parquet saved successfully.


In [34]:
import os

output_path = r"E:\celabal\spark-data-engineering-pipeline\output"

os.makedirs(output_path, exist_ok=True)

clean_df.toPandas().to_parquet(
    os.path.join(output_path, "superstore_output.parquet"),
    index=False
)

print("Parquet file saved successfully.")

Parquet file saved successfully.


In [35]:
import pandas as pd

parquet_df = pd.read_parquet(
    r"E:\celabal\spark-data-engineering-pipeline\output\superstore_output.parquet"
)

parquet_df.head(10)

,Order ID,Customer_Name,Category,Sub_Category,Sales,Quantity,Profit,City,State,Total_Amount,Order_Category
0,CA-2016-152156,Claire Gute,Furniture,Chairs,731.9400,3,219.5820,Henderson,Kentucky,2195.82,High Value
1,US-2015-108966,Sean O'Donnell,Furniture,Tables,957.5775,5,-383.0310,Fort Lauderdale,Florida,4787.89,High Value
2,CA-2014-115812,Brosina Hoffman,Technology,Phones,907.1520,6,90.7152,Los Angeles,California,5442.91,High Value
3,CA-2014-115812,Brosina Hoffman,Furniture,Tables,1706.1840,9,85.3092,Los Angeles,California,15355.66,High Value
4,CA-2014-115812,Brosina Hoffman,Technology,Phones,911.4240,4,68.3568,Los Angeles,California,3645.70,High Value
5,CA-2014-105893,Pete Kriz,Office Supplies,Storage,665.8800,6,13.3176,Madison,Wisconsin,3995.28,High Value
6,CA-2015-106320,Emily Burns,Furniture,Tables,1044.6300,3,240.2649,Orem,Utah,3133.89,High Value
7,US-2015-150630,Tracy Blumstein,Furniture,Bookcases,3083.4300,7,-1665.0522,Philadelphia,Pennsylvania,21584.01,High Value
8,CA-2016-117590,Gene Hale,Technology,Phones,1097.5440,7,123.4737,Richardson,Texas,7682.81,High Value
9,CA-2015-117415,Steve Nguyen,Furniture,Bookcases,532.3992,3,-46.9764,Houston,Texas,1597.20,High Value


In [36]:
spark.stop()

print("Spark session stopped successfully.")

Spark session stopped successfully.
